# v10-infer — chạy adapter b318 trên 500 câu BỊ LOẠI khỏi tập train

`build_sft.giu_mau()` loại 1.552/6.908 câu vì `khong_co_weak_gold` hoặc
`retrieval_truot_gold`, và docstring của nó đã ghi sẵn cái giá:

> model chưa từng thấy context thiếu gold, nên tỉ lệ bịa thật lúc inference cao hơn
> những gì dev set (lọc cùng cách) cho thấy.

Dev 120 câu chỉ chứa 21 mẫu như vậy — quá ít để đo. Kernel này chạy 500 mẫu lấy ngẫu
nhiên (seed 0) từ đúng 1.552 câu đó, có đáp án BTC nên chấm được.

`LEGALQA_SO_HIEU=0`: b318 được train trên prompt CHƯA có số hiệu văn bản. Suy luận bằng
prompt có số hiệu là thả model vào phân phối nó chưa từng thấy, và khi đó không phân
biệt được "model bịa" với "prompt lạ".


In [ ]:
import os, sys, shutil, zipfile, json, subprocess
from pathlib import Path

# Chỉ nhận codebase từ dataset `legalqa-v10-src`: `kernel_sources` mang theo `repo/` của
# lần chạy trước và glob bắt trúng bản CŨ đó trước.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
assert SRC, "Không thấy codebase — chưa gắn dataset legalqa-v10-src"

REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

# Ghim 5.14.1: model nền là `qwen3_5` (`Qwen3_5ForConditionalGeneration`), bản 5.0.0
# dựng sẵn của Kaggle không biết kiến trúc đó.
!pip -q install transformers==5.14.1 peft nltk
# Gỡ torchao 0.10.0 của Kaggle: peft mới đòi >0.16 và `dispatch_torchao` ném ImportError
# giữa lúc nạp adapter.
!pip -q uninstall -y torchao

os.chdir(REPO)
sys.path.insert(0, "src")
os.environ["PYTHONPATH"] = "src"
# Khớp prompt với checkpoint đang dùng, không phải với bản code mới nhất.
os.environ["LEGALQA_SO_HIEU"] = "0"

import torch, transformers
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())
print("transformers:", transformers.__version__)
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (5, 14), \
    "transformers quá cũ — cell pip chưa ăn"
print("GPU:", torch.cuda.get_device_name(0))
# Dataset Kaggle tạo version BẤT ĐỒNG BỘ — đẩy xong chạy ngay vẫn có thể nhận bản cũ.
# Hỏng trong 2 giây rẻ hơn hỏng sau khi đã tải reranker.
assert (REPO / "scripts/infer.py").exists(), \
    "chưa có scripts/infer.py — chờ legalqa-v10-src tạo xong version rồi Add Input lại"
from legalqa.prompt import CO_SO_HIEU
assert CO_SO_HIEU is False, "cờ LEGALQA_SO_HIEU không ăn — prompt sẽ lệch với b318"
print("số hiệu trong prompt:", CO_SO_HIEU)


In [ ]:
# clean/ vào bằng dataset `legalqa-clean`. Symlink sang working vì infer.py có thể ghi
# bm25.pkl, mà /kaggle/input read-only.
ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — chưa gắn dataset legalqa-clean"
CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    d = CLEAN / f.name
    if not d.exists():
        d.symlink_to(f)
os.environ["CLEAN_DIR"] = str(CLEAN)
for n in ("chunks.jsonl", "bm25.pkl", "corpus.jsonl"):
    assert (CLEAN / n).exists(), f"thiếu {n} trong legalqa-clean"

CAU_HOI = next(Path("/kaggle/input").glob("**/bo_qua_500.jsonl"), None)
assert CAU_HOI, "Không thấy bo_qua_500.jsonl — gắn dataset legalqa-bo-qua"
_r = [json.loads(l) for l in open(CAU_HOI, encoding="utf-8")]
from collections import Counter
print(CAU_HOI, len(_r), Counter(x["nhom"] for x in _r))
assert len(_r) == 500 and all(x.get("answer") for x in _r), "tập câu hỏi sai hình dạng"

# Model nền: tìm bằng tokenizer.json + "qwen" trong đường dẫn. Lọc theo "ForCausalLM"
# trong config.json là trượt — model nền là biến thể đa phương thức.
BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"

# Kaggle đổi tên shard khi upload; index.json vẫn trỏ tên chuẩn. Vá bằng symlink.
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/kaggle/working/base")
    FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX
    print("vá tên shard ->", BASE)

# Adapter b318 = `tot_nhat/` của phiên 2, vào bằng dataset legalqa-sft-phien2.
ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/tot_nhat/adapter_config.json")), None)
assert ADAPTER, "không thấy tot_nhat/ — gắn dataset legalqa-sft-phien2"

# Reranker: bản đã gắn nếu có, không thì tải (Internet ON).
local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    RERANK = str(local[0])
else:
    from huggingface_hub import snapshot_download
    RERANK = snapshot_download("AITeamVN/Vietnamese_Reranker")
print("base   ", BASE)
print("adapter", ADAPTER)
print("rerank ", RERANK)


In [ ]:
# subprocess.run(check=True), KHÔNG `!python`: trong IPython `!python` không ném theo mã
# thoát — script chết giữa chừng mà cell vẫn xanh và cell sau đọc file rỗng.
#
# Ngân sách: retrieval ~2s/câu (17 phút) + sinh ~12,7s/câu đo ở eval b318 (106 phút).
# Hai mốc giờ tách riêng để pha 1 không ăn mất giờ của pha 2 — cả hai pha đều ghi tăng
# dần ra jsonl nên chạy lại kernel là tiếp được, không mất phần đã sinh.
subprocess.run([sys.executable, "scripts/infer.py",
    "--cau-hoi", str(CAU_HOI),
    "--model", str(BASE), "--adapter", str(ADAPTER), "--rerank", RERANK,
    "--ra", "/kaggle/working/out",
    "--batch-size", "32", "--pool", "50",
    "--max-new", "1024", "--gen-bs", "4",
    "--gio-pha1", "1.5", "--gio-pha2", "6.0",
], check=True)


In [ ]:
# Nghiệm thu ĐÚNG thứ đang khẳng định: không phải "file có tồn tại" mà "model bịa bao
# nhiêu khi prompt thiếu căn cứ". So thẳng với 120 câu dev của cùng checkpoint b318
# (METEOR 0,5982 toàn cục; 0,6407 ở nhóm retrieval khớp gold).
import re
from collections import Counter

rows = [json.loads(l) for l in open("/kaggle/working/out/du_doan.jsonl", encoding="utf-8")]
pr = {json.loads(l)["id"]: json.loads(l)["prompt"]
      for l in open("/kaggle/working/out/prompts.jsonl", encoding="utf-8")}
NAM = re.compile(r"\b(?:19|20)\d{2}\b")
DIEU = re.compile(r"Điều\s+(\d+[a-zA-Z]?)")

st = Counter()
for x in rows:
    p, g, c = str(x["du_doan"]), str(x["dap_an"]), pr[x["id"]]
    pn, gn, cn = set(NAM.findall(p)), set(NAM.findall(g)), set(NAM.findall(c))
    pd, cd = set(DIEU.findall(p)), set(DIEU.findall(c))
    st["n"] += 1
    bia = pn - cn
    if bia:
        st["nam_bia"] += 1
        # "SAI" = có bịa, gold CÓ năm, mà không năm bịa nào trúng gold. Phải so trên
        # đúng tập bịa: so cả `pn` thì một năm chép đúng từ prompt cũng cứu cho câu bịa.
        if gn and not (bia & gn):
            st["nam_bia_SAI"] += 1
    if pd - cd:
        st["dieu_bia"] += 1
    if not p.rstrip().endswith((".", "!", "?", '"', ":", ";")):
        st["cut"] += 1
n = st["n"]
print(f"trên {n} câu bị loại khỏi train:")
for k in ("nam_bia", "nam_bia_SAI", "dieu_bia", "cut"):
    print(f"  {k:14} {st[k]:4}  {st[k]/n:6.1%}")
print("\nb318 trên 120 câu dev, cùng phép đo: nam_bia 65,0%  nam_bia_SAI 46,7%  "
      "dieu_bia 3,3%  cut 6,7%")
print(Counter(x["nhom"] for x in rows))
